In [ ]:
# ==========================================
# 1. Standard Library & Basic Imports
# ==========================================
import os
import re
import json
import asyncio
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import praw
import ast
import chromadb
import random
import nest_asyncio
import operator

# ==========================================
# 2. From Imports (Specific Modules)
# ==========================================

from dotenv import load_dotenv
from typing import Annotated, List, TypedDict
from datetime import datetime
from tqdm.auto import tqdm
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import AsyncOpenAI

import chromadb
from chromadb.utils import embedding_functions
from langchain_community.vectorstores import Chroma

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

nest_asyncio.apply()
load_dotenv()

In [ ]:
# ==========================================
# 3. Environment Setting
# ==========================================

# Reddit
client_id     = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")
user_agent    = os.getenv("USER_AGENT")

reddit = praw.Reddit(
    client_id     = client_id,
    client_secret = client_secret,
    user_agent    = user_agent
)

# LLM 모델
llm = ChatOpenAI(
    model        = "gpt-5.1",
    model_kwargs = {"service_tier": "flex"},
    temperature  = 0,
    api_key      = os.getenv("OPENAI_API_KEY"),
    response_format = {"type": "json_object"}
)

# 임베딩 모델
embeddings = OpenAIEmbeddings()

# ── 변경: ChromaDB를 Purpose별로 분리 ──
# 기존: 단일 컬렉션 → Professional/Casual CR이 섞여 RAG 오염
# 변경: Purpose마다 독립 컬렉션 → 세그먼트별 RAG 정확도 향상
PURPOSES   = ["Professional", "Casual"]
BATCH_SIZE = 50  # 한 번에 LLM에 전달할 리뷰 수

vector_dbs = {
    purpose: Chroma(
        collection_name    = f"cr_{purpose.lower()}",
        embedding_function = embeddings,
        persist_directory  = f"./chroma_db_{purpose.lower()}"
    )
    for purpose in PURPOSES
}

In [ ]:
# ==========================================
# 4. Class and Function Definitions
# ==========================================

# ── Phase 1 GraphState: 행별 Tagging (기존과 동일) ──
class TaggingState(TypedDict):
    raw_review:       str
    is_valid:         bool
    primary_purpose:  str
    designer_action:  str
    reasoning:        str
    purpose_signals:  List[str]
    evidence_snippet: str


# ── Phase 2 GraphState: 행별 CR Extraction ──
class ExtractionState(TypedDict):
    purpose:          str
    raw_review:       str
    designer_action:  str
    is_extractable:   bool          # Step 1: 이 리뷰에서 CR 추출 가능 여부
    cr_candidates:    List[str]     # Step 2: LLM이 추출한 CR 후보
    extracted_crs:    List[dict]    # Step 3: DB 매핑 후 최종 CR
    cr_inventory:     Annotated[List[str], operator.add]

# DB 유사도 임계값 (낮을수록 더 엄격하게 기존 용어 재사용)
SIMILARITY_THRESHOLD = 0.15


# --------------------------------------------------
# [Phase 1] Tagging (기존과 동일)
# --------------------------------------------------
def get_openai_tags(review_text):
    system_instruction = """
        ### Role: Senior NPD Research Data Auditor

        Your sole mission is to determine if a review is "Actionable Product Feedback"
        and classify its usage purpose.

        ## Step 1. Validity Check (is_valid):
        Do NOT look for noise patterns. Instead, ask yourself:

        "Can I complete this sentence from this review?
        'A product designer should [do X] because this user needs [Y].'"

        - If YES → proceed to Step 2
        - If NO  → is_valid: false, primary_purpose: "None", stop here

        A valid review must satisfy ALL of the following:
        1. It is a self-contained statement (not a reply fragment or conversation).
        2. It expresses a concrete preference, expectation, or complaint about
        the product's design, performance, or ergonomics.
        3. A product designer can extract a concrete requirement from it.
        The primary intent is NOT to seek help, ask for recommendations,
        or get answers from other users.

        ## Step 2. Evidence Collection (BEFORE assigning purpose):
        List the specific signals in the review that indicate Professional or Casual intent.

        - You MUST find at least 2 concrete, distinct signals to assign a purpose.
        - If you cannot find 2 signals, set primary_purpose: "None", is_valid: false.

        Signals for **Professional**:
            The laptop is used as a tool for specialized, skill-based work
            where performance directly impacts output quality or productivity.
            (e.g., software dev, data science, 4K editing, 3D rendering,
            heavy corporate multitasking, engineering tools, etc.)

        Signals for **Casual**:
            The laptop is used primarily for consumption, communication,
            or light tasks without domain-specific performance demands.
            (e.g., web browsing, streaming, social media,
            basic student homework, everyday home use, etc.)

        ## Step 3. Confidence Check:
        Ask yourself: "If I read this review 10 times, would I assign
        the same purpose every time with >90% certainty?"

        - If YES → assign the purpose
        - If NO, or if Professional and Casual signals are mixed/equal →
        primary_purpose: "None", is_valid: false

        ## Output Schema (JSON Only):
        Important: Fill in "purpose_signals" BEFORE deciding "primary_purpose".
        {
            "purpose_signals": ["signal 1 from the review", "signal 2 from the review"],
            "is_valid": boolean,
            "primary_purpose": "Professional" | "Casual" | "None",
            "designer_action": "Complete the sentence: 'A designer should [X] because this user needs [Y].' Leave empty string if is_valid is false.",
            "reasoning": "One sentence explaining the validity and purpose decision.",
            "evidence_snippet": "The most relevant quote. Empty string if invalid."
        }
    """
    try:
        response = llm.invoke([
            SystemMessage(content=system_instruction),
            HumanMessage(content=f'Review: "{review_text}"')
        ])
        return response.content
    except Exception as e:
        print(f"Error: {e}")
        return None


def tagging_node(state: TaggingState):
    print(f"\n--- [에이전트: Auditor/Tagger] 리뷰 분석 시작 ---")
    response_content = get_openai_tags(state["raw_review"])
    try:
        result = json.loads(
            response_content.replace("```json", "").replace("```", "").strip()
        )
        return {
            "is_valid":         result.get("is_valid", False),
            "primary_purpose":  result.get("primary_purpose", "None"),
            "designer_action":  result.get("designer_action", ""),
            "reasoning":        result.get("reasoning", ""),
            "purpose_signals":  result.get("purpose_signals", []),
            "evidence_snippet": result.get("evidence_snippet", "")
        }
    except Exception as e:
        print(f"Tagging 노드 결과 파싱 중 오류 발생: {e}")
        return {
            "is_valid": False, "primary_purpose": "None",
            "designer_action": "", "reasoning": f"JSON Parsing Error: {str(e)}",
            "purpose_signals": [], "evidence_snippet": ""
        }


def should_continue(state: TaggingState):
    return "continue" if state["is_valid"] else "end"


# --------------------------------------------------
# [Phase 2] CR Extraction - 3단계 구조
# --------------------------------------------------

# Step 1: 이 리뷰에서 CR을 추출할 수 있는지 판단
def extractability_check_node(state: ExtractionState):
    """
    리뷰 1개를 받아서 CR 추출 가능 여부만 판단합니다.
    추출 가능한 경우에만 다음 단계로 넘어갑니다.
    """
    purpose        = state["purpose"]
    review         = state["raw_review"]
    designer_action = state["designer_action"]

    system_prompt = f"""
        You are a QFD data analyst for the "{purpose}" user segment.
        Determine if the following review contains an extractable Customer Requirement (CR).

        A CR is extractable if:
        - The review expresses a specific, concrete product need or complaint.
        - The need can be translated into a noun-phrase label for a QFD matrix.

        A CR is NOT extractable if:
        - The review is too vague, too short, or only expresses general sentiment.
        - The review is a question, a recommendation request, or off-topic.

        Design guidance: "{designer_action}"

        Output Schema (JSON only):
        {{
            "is_extractable": boolean,
            "reason": "One sentence explaining why."
        }}
    """
    try:
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f'Review: "{review}"')
        ])
        result = json.loads(
            response.content.replace("```json", "").replace("```", "").strip()
        )
        is_extractable = result.get("is_extractable", False)
        print(f"  [Step 1] Extractable: {is_extractable}")
        return {"is_extractable": is_extractable}
    except Exception as e:
        print(f"  [Step 1 Error] {e}")
        return {"is_extractable": False}


def should_extract(state: ExtractionState):
    return "extract" if state["is_extractable"] else "end"


# Step 2: CR 후보 추출
def extraction_node(state: ExtractionState):
    """
    추출 가능한 리뷰에서 CR 후보 레이블만 추출합니다.
    DB 매핑은 다음 단계(db_mapping_node)에서 처리합니다.
    """
    purpose         = state["purpose"]
    review          = state["raw_review"]
    designer_action = state["designer_action"]

    print(f"  [Step 2] CR 후보 추출 중...")

    system_prompt = f"""
        ### Identity: The Voice of the "{purpose}" User Group
        You are the representative of the "{purpose}" community.

        ### Design Guidance: "{designer_action}"

        ### Mission:
        Extract Atomic Customer Requirements (CR) from the following single review.

        ### CR Naming Style: [Direction adjective] + [Attribute noun] (+ optional spec)
        The CR must convey WHAT the user wants AND in WHICH DIRECTION.
        A bare attribute name like "Battery life" or "Weight" is NOT acceptable.

        ### Examples:
        - ❌ Too vague:    "Battery life"
        - ❌ Too verbose:  "Long battery life for all-day mobile work"
        - ✅ Good:         "Extended battery life"
        - ✅ Good:         "Battery life (all-day)"

        - ❌ Too vague:    "Weight"
        - ✅ Good:         "Reduced chassis weight"

        - ❌ Too vague:    "Thermal management"
        - ✅ Good:         "Stable thermal performance under sustained load"
        - ✅ Good:         "Low fan noise during operation"

        ### Additional Rules:
        - NO comparative expressions, reasons, or planner language
        - Split multi-need reviews into separate atomic CRs
        - Extract ONLY clearly stated needs. Maximum 3 CRs per review.

        ### Output Schema (JSON only):
        {{
            "cr_candidates": [
                {{
                    "label": "Direction + Attribute noun-phrase",
                    "evidence": "Direct quote or paraphrase from the review"
                }}
            ]
        }}
    """
    try:
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f'Review: "{review}"')
        ])
        result = json.loads(
            response.content.replace("```json", "").replace("```", "").strip()
        )
        candidates = result.get("cr_candidates", [])
        print(f"  [Step 2] {len(candidates)}개 후보 추출")
        return {"cr_candidates": candidates}
    except Exception as e:
        print(f"  [Step 2 Error] {e}")
        return {"cr_candidates": []}


# Step 3: DB 유사도 비교 → 기존 용어 재사용 or 신규 등록
def db_mapping_node(state: ExtractionState):
    """
    각 CR 후보를 DB와 비교합니다.
    - 유사도 높음 (distance < threshold): 기존 표준 용어 그대로 사용
    - 유사도 낮음 (distance >= threshold): 신규 CR로 확정 후 DB 저장
    """
    purpose     = state["purpose"]
    candidates  = state["cr_candidates"]
    db          = vector_dbs[purpose]
    final_crs   = []

    print(f"  [Step 3] DB 매핑 중... (threshold={SIMILARITY_THRESHOLD})")

    for candidate in candidates:
        label    = candidate.get("label", "")
        evidence = candidate.get("evidence", "")
        if not label:
            continue

        try:
            # DB가 비어있으면 similarity_search_with_score가 오류날 수 있음
            results = db.similarity_search_with_score(label, k=1)

            if results and results[0][1] < SIMILARITY_THRESHOLD:
                # 기존 용어 재사용
                existing_term = results[0][0].page_content
                distance      = results[0][1]
                print(f"    ✓ 매핑: '{label}' → '{existing_term}' (dist={distance:.3f})")
                final_crs.append({
                    "requirement":          existing_term,
                    "is_mapped_from_memory": True,
                    "distance":             round(distance, 3),
                    "evidence_summary":     evidence,
                    "purpose":              purpose
                })
            else:
                # 신규 CR 등록
                db.add_texts(
                    [label],
                    metadatas=[{"purpose": purpose}]
                )
                dist_str = f"{results[0][1]:.3f}" if results else "N/A (empty DB)"
                print(f"    + 신규: '{label}' (dist={dist_str})")
                final_crs.append({
                    "requirement":          label,
                    "is_mapped_from_memory": False,
                    "distance":             results[0][1] if results else None,
                    "evidence_summary":     evidence,
                    "purpose":              purpose
                })

        except Exception as e:
            # DB 비어있는 초기 상태 등 예외 처리
            print(f"    + 신규 (DB 초기): '{label}'")
            db.add_texts([label], metadatas=[{"purpose": purpose}])
            final_crs.append({
                "requirement":          label,
                "is_mapped_from_memory": False,
                "distance":             None,
                "evidence_summary":     evidence,
                "purpose":              purpose
            })

    return {
        "extracted_crs": final_crs,
        "cr_inventory":  [cr["requirement"] for cr in final_crs]
    }


In [ ]:
# ==========================================
# 5. Workflow Construction
# ==========================================

# ── Phase 1 Workflow: Tagging (행별, 기존과 동일) ──
tagging_workflow = StateGraph(TaggingState)
tagging_workflow.add_node("Tagger", tagging_node)
tagging_workflow.set_entry_point("Tagger")
tagging_workflow.add_conditional_edges(
    "Tagger",
    should_continue,
    {"continue": END, "end": END}
)
tagging_app = tagging_workflow.compile()

# ── Phase 2 Workflow: CR Extraction (행별, 3단계) ──
# Step 1: Extractability Check
# Step 2: CR Candidate Extraction  (추출 가능할 때만)
# Step 3: DB Mapping               (기존 용어 재사용 or 신규 등록)
extraction_workflow = StateGraph(ExtractionState)
extraction_workflow.add_node("ExtractabilityCheck", extractability_check_node)
extraction_workflow.add_node("Extraction",          extraction_node)
extraction_workflow.add_node("DBMapping",           db_mapping_node)

extraction_workflow.set_entry_point("ExtractabilityCheck")
extraction_workflow.add_conditional_edges(
    "ExtractabilityCheck",
    should_extract,
    {"extract": "Extraction", "end": END}
)
extraction_workflow.add_edge("Extraction", "DBMapping")
extraction_workflow.add_edge("DBMapping",  END)
extraction_app = extraction_workflow.compile()


In [ ]:
raw_data = pd.read_excel('../Data/raw_data/reddit_reviews_03.xlsx')
raw_data = raw_data[:1000]
raw_data.shape

In [ ]:
tagging_app

In [ ]:
extraction_app

In [ ]:
raw_data.head(3)

In [ ]:
# ==========================================
# 6. Execution
# ==========================================

# ── Phase 1: Tagging (행별 처리, 기존과 동일) ──
tagging_results = []

for i, review in enumerate(tqdm(raw_data['Body'], desc="[Phase 1] Tagging")):
    try:
        final_state = tagging_app.invoke({"raw_review": review})
        tagging_results.append({
            "Index":           i,
            "Original_Review": review,
            "Is_Valid":        final_state.get("is_valid", False),
            "Purpose":         final_state.get("primary_purpose", "None"),
            "Reasoning":       final_state.get("reasoning", ""),
            "Designer_Action": final_state.get("designer_action", ""),
            "Evidence":        final_state.get("evidence_snippet", "")
        })
    except Exception as e:
        print(f"\n[Error] 인덱스 {i} 처리 중 중단: {e}")
        tagging_results.append({
            "Index": i, "Original_Review": review,
            "Is_Valid": False, "Purpose": "None",
            "Reasoning": f"Error: {e}",
            "Designer_Action": "", "Evidence": ""
        })

tagging_df = pd.DataFrame(tagging_results)

print("\n[Phase 1 완료] Purpose 분포:")
print(tagging_df[tagging_df['Is_Valid'] == True]['Purpose'].value_counts())

In [ ]:
# ── Phase 2: CR Extraction (행별 처리, 3단계) ──
cr_results = {purpose: [] for purpose in PURPOSES}

for purpose in PURPOSES:
    segment_df = tagging_df[
        (tagging_df["Is_Valid"] == True) &
        (tagging_df["Purpose"] == purpose)
    ].reset_index(drop=True)

    total = len(segment_df)
    print(f"\n[Phase 2] {purpose} 세그먼트: {total}개 리뷰")

    if total == 0:
        print(f"  → valid 리뷰 없음, 스킵")
        continue

    for i, row in enumerate(tqdm(segment_df.itertuples(), total=total, desc=f"{purpose}")):
        try:
            final_state = extraction_app.invoke({
                "purpose":         purpose,
                "raw_review":      row.Original_Review,
                "designer_action": row.Designer_Action,
                "is_extractable":  False,
                "cr_candidates":   [],
                "extracted_crs":   [],
                "cr_inventory":    []
            })
            extracted = final_state.get("extracted_crs", [])
            if extracted:
                cr_results[purpose].extend(extracted)
        except Exception as e:
            print(f"  [Error] Index {row.Index}: {e}")
            continue

    print(f"  → {purpose} 완료: 총 {len(cr_results[purpose])}개 CR")

print("\n[Phase 2 완료]")
for p, crs in cr_results.items():
    print(f"  {p}: {len(crs)}개")


In [ ]:
tagging_df.head(10)

In [ ]:
# ==========================================
# 7. 결과 저장
# ==========================================

# Phase 1: Tagging 결과
tagging_df.to_excel(
    '../Data/CR_Extraction/phase1_tagging_result.xlsx',
    index=False
)

# Phase 2: CR Extraction 결과 (Purpose별 시트 분리)
cr_output_path = '../Data/CR_Extraction/phase2_cr_extraction_v2.xlsx'
with pd.ExcelWriter(cr_output_path, engine='openpyxl') as writer:
    for purpose, crs in cr_results.items():
        if crs:
            cr_df = pd.DataFrame(crs)
            if "supporting_reviews" in cr_df.columns:
                cr_df["supporting_reviews"] = cr_df["supporting_reviews"].apply(
                    lambda x: ", ".join(x) if isinstance(x, list) else str(x)
                )
            cr_df.to_excel(writer, sheet_name=purpose, index=False)

print(f"저장 완료: {cr_output_path}")

In [ ]:
# ChromaDB 확인
for purpose in PURPOSES:
    print(f"chroma_db_{purpose.lower()}: {os.path.exists(f'./chroma_db_{purpose.lower()}')}")

In [ ]:
# ==========================================
# 8. Phase 3 - CR Consolidation
#    배치 간 중복 CR을 LLM으로 최종 통합
#    목표: 세그먼트당 10~15개로 압축
# ==========================================

CONSOLIDATION_PROMPT = """
### Role: QFD Data Curator
You will receive a raw list of Customer Requirements (CRs) extracted
from multiple review batches for the "{purpose}" user segment.
Some CRs may be duplicates or near-duplicates across batches.

### Task:
1. **Merge duplicates**: If two or more CRs represent the same need,
   merge them into one canonical noun-phrase term.
   Use the most concise and standard phrasing.
2. **Keep distincts**: If two CRs represent genuinely different needs, keep both.
3. **Count Limit**: The final list must contain 10 to 15 CRs maximum.
   If you have more, keep only the most representative ones.
   Prioritize CRs that were mentioned across multiple batches.

### Output Schema (JSON only):
{{
    "consolidated_crs": [
        {{
            "requirement": "Final canonical noun-phrase label",
            "merged_from": ["original CR 1", "original CR 2"],
            "evidence_summary": "Aggregated summary"
        }}
    ]
}}
"""


def consolidate_crs(purpose: str, raw_crs: list) -> list:
    if not raw_crs:
        return []

    cr_list_text = "\n".join(
        [f"- {cr['requirement']}" for cr in raw_crs]
    )

    try:
        response = llm.invoke([
            SystemMessage(content=CONSOLIDATION_PROMPT.format(purpose=purpose)),
            HumanMessage(content=f"Raw CR list:\n{cr_list_text}")
        ])
        result = json.loads(
            response.content.replace("```json", "").replace("```", "").strip()
        )
        consolidated = result.get("consolidated_crs", [])
        for cr in consolidated:
            cr["purpose"] = purpose
        return consolidated
    except Exception as e:
        print(f"[Consolidation Error / {purpose}] {e}")
        return raw_crs


# 실행
consolidated_results = {}

for purpose in PURPOSES:
    raw_count = len(cr_results[purpose])
    print(f"[Phase 3] {purpose}: {raw_count}개 → 통합 중...")
    consolidated_results[purpose] = consolidate_crs(purpose, cr_results[purpose])
    print(f"  → {len(consolidated_results[purpose])}개로 정리")

# 저장
consolidated_path = '../Data/CR_Extraction/phase3_cr_consolidated_v3.xlsx'
with pd.ExcelWriter(consolidated_path, engine='openpyxl') as writer:
    for purpose, crs in consolidated_results.items():
        if crs:
            df = pd.DataFrame(crs)
            if "merged_from" in df.columns:
                df["merged_from"] = df["merged_from"].apply(
                    lambda x: ", ".join(x) if isinstance(x, list) else str(x)
                )
            df.to_excel(writer, sheet_name=purpose, index=False)

print(f"\n저장 완료: {consolidated_path}")

In [ ]:
# 최종 결과 확인 (Consolidation 기준)
for purpose, crs in consolidated_results.items():
    print(f"\n{'='*40}")
    print(f"  {purpose} ({len(crs)}개 CR)")
    print(f"{'='*40}")
    for i, cr in enumerate(crs, 1):
        print(f"  [{i:02d}] {cr['requirement']}")
        if cr.get('evidence_summary'):
            print(f"       └ {cr['evidence_summary'][:80]}")